## Running this notebook

Install the project with the notebook extra before opening this file. Imports
use the installed package and do not change the working directory. Set
`OVERSCREENING_OUTPUT_DIR` to choose a directory for saved files.

The original scientific parameter values and sweep sizes are retained. Several
cells are deliberately expensive or demonstrate a failed solve; they are not
part of the bounded CI examples. Inspect convergence flags before interpreting
results. Adjoint initialization and unsupported legacy boundary/operator
experiments are explicitly labelled and use the canonical collocation path.
For bounded NumPy/mpmath demonstrations, run the scripts in `examples/`.

This notebook intentionally retains legacy nonstandard operators and Neumann
wall conditions. Those cases are collocation-only; they are not examples of
the mixed physical flux formulation.


In [ ]:
#load modules
from matplotlib import pyplot as plt
import numpy as np
from overscreening_breakdown.collocation.spectral1d import solve_nonlinear_problem # needed module
from overscreening_breakdown import overscreening_breakdown #needed module
from overscreening_breakdown.manufactured.legacy_additional import \
problem_discontinuous_lhs_1, \
problem_discontinuous_lhs_2, \
problem_discontinuous_lhs_3, \
problem_discontinuous_lhs_4, \
problem_neumann_bc_at_0, \
problem_neumann_bc_at_0_der_4_and_2, \
problem_neumann_bc_at_0_der_4_and_2_2

%matplotlib inline

In [ ]:
all_problems = [problem_discontinuous_lhs_1(), \
                problem_discontinuous_lhs_2(), \
                problem_discontinuous_lhs_3(), \
                problem_discontinuous_lhs_4(), \
                problem_neumann_bc_at_0(), \
                problem_neumann_bc_at_0_der_4_and_2(), \
                problem_neumann_bc_at_0_der_4_and_2_2()
               ]

# initial distribution to make things harder for the solver.
def some_func(x):
    return 1/(1+x)

for j in range(0,len(all_problems)):
    all_problems[j].set_parameters(initial_guess_solution=some_func)

In [ ]:
N = 100
domain = [0,np.inf]
tolerance = 1.0e-10
use_method = "newton"
use_adjoint_opimization = False
use_globalization = False
L = 6 #on the sellection of this parameter, see bellow 

solver = solve_nonlinear_problem(N=N, 
                                 domain=domain, 
                                 tolerance=tolerance, 
                                 use_method=use_method, 
                                 use_adjoint_opimization = use_adjoint_opimization, 
                                 use_globalization = use_globalization)
solver.set_mapping_parameter(L)

In [ ]:
def plot_all(c, problem):
    #plotting in computational domain in [0, $pi$]
    t = np.arange(0, np.pi, 0.01) 
    sol = solver.obtain_solution_in_basis(c, t)
    plt.title(r"solution on the mapped segment [0,$\pi$]")
    plt.plot(t, sol,'.')
    plt.legend([problem.info()])
    plt.show()

    #plotting in physical domain for 0<=x<=100
    x = np.arange(0, 10, 0.005)
    sol = solver.obtain_solution_in_domain(c, x)
    plt.title(r"solution on the part of the $\mathbb{R}^+$")
    plt.plot(x, sol, '.')
    plt.legend([problem.info()])
    plt.show()


    #plotting the rhs of the problem
    x = np.arange(0.0,10, 0.01)
    sol = solver.obtain_solution_in_domain(c, x)
    rhs = problem.right_hand_side(x,sol,0)
    plt.title(r"rhs on the part of the $\mathbb{R}^+$")
    plt.plot(x, rhs, '.')
    plt.legend([problem.info()])
    plt.show()    

In [ ]:
#testing a particular problem number
problem = all_problems[6]
problem.info()
problem.get_name()
solver.set_problem(problem)
c = solver.solve_problem()
plot_all(c, problem)

In [ ]:
#testing all problems
for problem in all_problems:
    print(problem.info())
    print(problem.get_name())
    solver.set_problem(problem)
    c = solver.solve_problem()
    plot_all(c, problem)
print("done.")